## Setup

In [1]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [2]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [3]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [4]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='el')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [5]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-ORG', 'I-PER', 'O', 'B-LOC', 'I-ORG', 'B-PER', 'I-LOC'}


# Evaluate model

In [7]:
alignment = {
'B-LOC': 'B-LOC',
'B-MISC': 'O',
'B-ORG': 'B-ORG',
'I-LOC': 'I-LOC',
'I-MISC': 'O',
'I-ORG': 'I-ORG',
'I-PER': 'I-PER',
'O': 'O'
}

model_name = "xlm-roberta-large-finetuned-conll03-english"
model_name_output = 'xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at xlm-roberta-large-finetuned-conll03-english were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# model_evaluation.model.config.id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'I-LOC',
 4: 'I-MISC',
 5: 'I-ORG',
 6: 'I-PER',
 7: 'O'}

### wikiann

In [8]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [9]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6554,0.8239,0.7301,4476
1,ORG,0.6530,0.5067,0.5706,3655
2,PER,0.6627,0.7374,0.6980,4033
3,micro,0.6574,0.6999,0.6780,12164
4,macro,0.6570,0.6894,0.6663,12164
5,weighted,0.6571,0.6999,0.6715,12164


In [10]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0000,0.0000,0.0000,4476
1,B-ORG,0.0000,0.0000,0.0000,3655
2,B-PER,0.0000,0.0000,0.0000,4032
3,I-LOC,0.1555,0.5779,0.2450,1919
4,I-ORG,0.4702,0.4154,0.4411,6685
5,I-PER,0.4400,0.6678,0.5305,5392
6,O,0.9112,0.9809,0.9448,64507
7,accuracy,0.7805,90666,None,None
8,macro,0.2824,0.3774,0.3088,90666
9,weighted,0.7124,0.7805,0.7415,90666
